In [ ]:
%pip install numpy
%pip install pandas
%pip install matplotlib
%pip install seaborn
%pip install pathlib
%pip install scikit-learn
%pip install imbalanced-learn
%pip install scipy

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

In [1]:
import os
import multiprocessing

num_cores = multiprocessing.cpu_count()
print(f"La máquina tiene {num_cores} núcleos disponibles.")
hilos_optimos = str(min(num_cores, 64)) 
print(f"Configurando OpenBLAS para usar {hilos_optimos} hilos máximos...")
os.environ['OPENBLAS_NUM_THREADS'] = hilos_optimos
os.environ['MKL_NUM_THREADS'] = hilos_optimos
os.environ['OMP_NUM_THREADS'] = hilos_optimos

La máquina tiene 224 núcleos disponibles.
Configurando OpenBLAS para usar 64 hilos máximos...


In [2]:
import numpy as np
import pandas as pd
import itertools
import json
from pathlib import Path
from collections import Counter
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif, SelectKBest
from sklearn.svm import SVC
from sklearn.metrics import (f1_score, accuracy_score, recall_score,classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from imblearn.combine import SMOTEENN


In [3]:
X = pd.read_pickle("Sets_Xy/X.pkl")
y = pd.read_pickle("Sets_Xy/y.pkl")

In [4]:
from sklearn.model_selection import train_test_split

#Division estratificada para muestras de cada clase a nivel de cada subset

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, 
    random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, 
    random_state=42)

In [5]:
#Encoding de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)
y_test = le.transform(y_test)

mapeo_labels = pd.DataFrame({
    "label_original": le.classes_,
    "label_encoded": range(len(le.classes_))
})
print("Mapeo de etiquetas:\n", mapeo_labels)
class_names = le.classes_

Mapeo de etiquetas:
                label_original  label_encoded
0                      BENIGN              0
1                         Bot              1
2                        DDoS              2
3               DoS GoldenEye              3
4                    DoS Hulk              4
5            DoS Slowhttptest              5
6               DoS slowloris              6
7                 FTP-Patator              7
8                  Heartbleed              8
9                Infiltration              9
10                   PortScan             10
11                SSH-Patator             11
12    Web Attack  Brute Force             12
13  Web Attack  Sql Injection             13
14            Web Attack  XSS             14


In [11]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import ExtraTreesClassifier

class FeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, method='tree', k=50, class_weight='balanced', random_state=42):
        self.method = method
        self.k = k
        self.class_weight = class_weight
        self.random_state = random_state
        self.selector = None
        self.feature_names_ = None
        self.support_mask_ = None

    def fit(self, X, y):
        self.feature_names_ = X.columns.tolist() if hasattr(X, 'columns') else list(range(X.shape[1]))
        
        X_array = X.values if hasattr(X, 'values') else X
        y_array = y.values if hasattr(y, 'values') else y
        
        total_features = X_array.shape[1]
        actual_k = min(self.k, total_features)
        
        if self.method == 'none':
            self.support_mask_ = np.ones(total_features, dtype=bool)
            return self

        if self.method == 'f_classif':
            self.selector = SelectKBest(score_func=f_classif, k=actual_k)
            self.selector.fit(X_array, y_array)
            self.support_mask_ = self.selector.get_support()

        elif self.method == 'tree':
            estimator = ExtraTreesClassifier(
                n_estimators=100,               
                max_depth=20,                   
                class_weight=self.class_weight,
                criterion='gini',               
                random_state=self.random_state, 
                n_jobs=-1
            )
            estimator.fit(X_array, y_array)
            importances = estimator.feature_importances_
            
            top_k_indices = np.argsort(importances)[-actual_k:]
            
            self.support_mask_ = np.zeros(total_features, dtype=bool)
            self.support_mask_[top_k_indices] = True

        return self

    def transform(self, X):
        if self.method == 'none':
            return X
            
        X_array = X.values if hasattr(X, 'values') else X
        X_transformed = X_array[:, self.support_mask_]

        if hasattr(X, 'columns'):
            selected_features = np.array(self.feature_names_)[self.support_mask_].tolist()
            return pd.DataFrame(X_transformed, columns=selected_features, index=X.index)
        else:
            return X_transformed

In [10]:
#Carga de datos post oversampling
import joblib

X_train_none = joblib.load("Sets_Oversampling/X_train_none.joblib")
y_train_none = joblib.load("Sets_Oversampling/y_train_none.joblib")

X_train_smote = joblib.load("Sets_Oversampling/X_train_smote.joblib")
y_train_smote = joblib.load("Sets_Oversampling/y_train_smote.joblib")

X_train_smote_enn = joblib.load("Sets_Oversampling/X_train_smote_enn.joblib")
y_train_smote_enn = joblib.load("Sets_Oversampling/y_train_smote_enn.joblib")

X_train_smote_tomek = joblib.load("Sets_Oversampling/X_train_smote_tomek.joblib")
y_train_smote_tomek = joblib.load("Sets_Oversampling/y_train_smote_tomek.joblib")

X_val = pd.read_pickle("Sets_Post_Scaled_Imp/X_val_scaled_bayesian.pkl")
y_val = pd.DataFrame(y_val)

X_test = pd.read_pickle("Sets_Post_Scaled_Imp/X_test_scaled_bayesian.pkl")
y_test = pd.DataFrame(y_test)

In [11]:
X_train_none_2 = joblib.load("Sets_Oversampling_2/X_train_none.joblib")
y_train_none_2 = joblib.load("Sets_Oversampling_2/y_train_none.joblib")

X_train_smote_2 = joblib.load("Sets_Oversampling_2/X_train_smote.joblib")
y_train_smote_2 = joblib.load("Sets_Oversampling_2/y_train_smote.joblib")

X_train_smote_enn_2 = joblib.load("Sets_Oversampling_2/X_train_smote_enn.joblib")
y_train_smote_enn_2 = joblib.load("Sets_Oversampling_2/y_train_smote_enn.joblib")

X_train_smote_tomek_2 = joblib.load("Sets_Oversampling_2/X_train_smote_tomek.joblib")
y_train_smote_tomek_2 = joblib.load("Sets_Oversampling_2/y_train_smote_tomek.joblib")


In [16]:
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

def balanced_class_weight(y):
    classes = np.unique(y)
    weights = compute_class_weight('balanced', classes=classes, y=y)
    return dict(zip(classes, weights))

def polynomial_class_weight(y, alpha=0.25):
    classes, counts = np.unique(y, return_counts=True)
    weights = 1.0 / np.power(counts, alpha)
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

def log_ratio_class_weight(y):
    classes, counts = np.unique(y, return_counts=True)
    total = np.sum(counts)
    weights = np.log(total / counts)
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

def effective_samples_class_weight(y, beta=0.999):
    classes, counts = np.unique(y, return_counts=True)
    effective_num = 1.0 - np.power(beta, counts)
    weights = (1.0 - beta) / effective_num
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

def cost_sensitive_class_weight(y):
    classes, counts = np.unique(y, return_counts=True)
    max_count = np.max(counts)
    weights = max_count / counts
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

def focal_class_weight_improved(y, gamma=2.0):

    classes, counts = np.unique(y, return_counts=True)
    probs = counts / np.sum(counts)
    weights = (1 - probs) ** gamma / probs
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

In [51]:
import os
import logging
import joblib
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import optuna

os.makedirs("Logs_RF_Baseline", exist_ok=True)

logging.basicConfig(
    filename='Logs_RF_Baseline/optuna_rf_log.txt',
    level=logging.INFO,
    format='%(message)s'
)
logger = logging.getLogger()

In [13]:
train_datasets = {
    'none': (X_train_none, y_train_none.values.ravel() if isinstance(y_train_none, pd.DataFrame) else y_train_none.ravel()),
    'smote_enn': (X_train_smote_enn, y_train_smote_enn.values.ravel() if isinstance(y_train_smote_enn, pd.DataFrame) else y_train_smote_enn.ravel()),
    'smote_tomek': (X_train_smote_tomek, y_train_smote_tomek.values.ravel() if isinstance(y_train_smote_tomek, pd.DataFrame) else y_train_smote_tomek.ravel())
}

y_val_1d = y_val.values.ravel() if isinstance(y_val, pd.DataFrame) else y_val.ravel()
y_test_1d = y_test.values.ravel() if isinstance(y_test, pd.DataFrame) else y_test.ravel()

In [14]:
train_datasets_2 = {
    'none': (X_train_none_2, y_train_none_2.values.ravel() if isinstance(y_train_none_2, pd.DataFrame) else y_train_none_2.ravel()),
    'smote': (X_train_smote_2, y_train_smote_2.values.ravel() if isinstance(y_train_smote_2, pd.DataFrame) else y_train_smote_2.ravel()),
    'smote_enn': (X_train_smote_enn_2, y_train_smote_enn_2.values.ravel() if isinstance(y_train_smote_enn_2, pd.DataFrame) else y_train_smote_enn_2.ravel()),
    'smote_tomek': (X_train_smote_tomek_2, y_train_smote_tomek_2.values.ravel() if isinstance(y_train_smote_tomek_2, pd.DataFrame) else y_train_smote_tomek_2.ravel())
}

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
import optuna

os.makedirs("Logs_RF_Baseline", exist_ok=True)

def save_confusion_matrix(y_true, y_pred, trial_number, phase="Val"):
    """Guarda la matriz de confusión como imagen."""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - Trial {trial_number} ({phase})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'Logs_RF_Baseline/Trial_{trial_number}_RF_{phase}_CM.png')
    plt.close()

def objective_rf(trial):
    dataset_name = trial.suggest_categorical('dataset', ['none', 'smote_enn', 'smote_tomek'])
    fs_method = trial.suggest_categorical('feature_selection', ['none', 'f_classif'])
    
    X_train_raw, y_train_raw = train_datasets[dataset_name]
    
    selector = FeatureSelector(method=fs_method, k=50) 
    X_train_fs = selector.fit_transform(X_train_raw, y_train_raw)
    X_val_fs = selector.transform(X_val)
    
    weight_func_name = trial.suggest_categorical('weight_func', ['polynomial', 'effective_samples'])
    if weight_func_name == 'polynomial':
        alpha = trial.suggest_float('poly_alpha', 0.1, 0.9)
        weight_dict = polynomial_class_weight(y_train_raw, alpha=alpha)
    else:
        weight_dict = effective_samples_class_weight(y_train_raw)
    
    rf_params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 10, 50),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 15),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'class_weight': weight_dict,
        'n_jobs': -1,
        'random_state': 42
    }
    
    model = RandomForestClassifier(**rf_params)
    model.fit(X_train_fs, y_train_raw)
    y_pred = model.predict(X_val_fs)
    
    f1_mac = f1_score(y_val_1d, y_pred, average='macro')
    report = classification_report(y_val_1d, y_pred, zero_division=0)
    
    save_confusion_matrix(y_val_1d, y_pred, trial.number, phase="Val")
    
    log_msg = f"""Trial {trial.number}
Dataset: {dataset_name} | Feature Selection: {fs_method}
Params RF: {trial.params}

Metricas Clave
F1 Macro (Validation): {f1_mac:.4f}

Classification Report (Precision, Recall, F1 por clase)
{report}
"""
    
    log_filename = f"Logs_RF_Baseline/Trial_{trial.number}_Metrics.log"
    with open(log_filename, 'w', encoding='utf-8') as f:
        f.write(log_msg)
    
    print(f"Trial {trial.number} completado | F1 Macro: {f1_mac:.4f} | Log guardado en {log_filename}")
    
    return f1_mac

print("Iniciando búsqueda con Optuna para Random Forest...")
study_rf = optuna.create_study(direction='maximize', study_name="RandomForest_IDS_Optimization")

study_rf.optimize(objective_rf, n_trials=50, n_jobs=2) 

print(f"Mejor Trial: {study_rf.best_trial.number}")
print(f"Mejor F1 Macro: {study_rf.best_value:.4f}")
print(f"Mejores Hiperparametros:\n{study_rf.best_params}")

[I 2026-03-07 19:49:23,138] A new study created in memory with name: RandomForest_IDS_Optimization


Iniciando búsqueda con Optuna para Random Forest...


[I 2026-03-07 19:49:56,995] Trial 0 finished with value: 0.8331624420185365 and parameters: {'dataset': 'none', 'feature_selection': 'f_classif', 'weight_func': 'polynomial', 'poly_alpha': 0.37693061026287644, 'n_estimators': 160, 'max_depth': 37, 'min_samples_split': 9, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 0 with value: 0.8331624420185365.


Trial 0 completado | F1 Macro: 0.8332 | Log guardado en Logs_RF_Baseline/Trial_0_Metrics.log


[I 2026-03-07 19:50:13,489] Trial 6 finished with value: 0.8358995280740512 and parameters: {'dataset': 'none', 'feature_selection': 'none', 'weight_func': 'effective_samples', 'n_estimators': 154, 'max_depth': 45, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_features': None}. Best is trial 6 with value: 0.8358995280740512.


...

Trial 47 completado | F1 Macro: 0.8299 | Log guardado en Logs_RF_Baseline/Trial_47_Metrics.log


[I 2026-03-07 20:16:16,757] Trial 48 finished with value: 0.8004612841716516 and parameters: {'dataset': 'smote_tomek', 'feature_selection': 'f_classif', 'weight_func': 'effective_samples', 'n_estimators': 75, 'max_depth': 33, 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 42 with value: 0.865709881973818.


Trial 48 completado | F1 Macro: 0.8005 | Log guardado en Logs_RF_Baseline/Trial_48_Metrics.log


[I 2026-03-07 20:18:13,337] Trial 49 finished with value: 0.8326453890781783 and parameters: {'dataset': 'smote_tomek', 'feature_selection': 'f_classif', 'weight_func': 'effective_samples', 'n_estimators': 65, 'max_depth': 33, 'min_samples_split': 3, 'min_samples_leaf': 7, 'max_features': None}. Best is trial 42 with value: 0.865709881973818.


Trial 49 completado | F1 Macro: 0.8326 | Log guardado en Logs_RF_Baseline/Trial_49_Metrics.log
Mejor Trial: 42
Mejor F1 Macro: 0.8657
Mejores Hiperparametros:
{'dataset': 'smote_tomek', 'feature_selection': 'f_classif', 'weight_func': 'effective_samples', 'n_estimators': 93, 'max_depth': 35, 'min_samples_split': 4, 'min_samples_leaf': 3, 'max_features': 'sqrt'}


In [61]:
#Para mejorar el approach voy a hacer que el k del feature selection sea un hiperparametro y voy a hacer trials de manera independiente por dataset

import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
import optuna

os.makedirs("Logs_RF_Baseline_2", exist_ok=True)

class FeatureSelector:
    def __init__(self, method='f_classif', k=50):
        self.method = method
        self.k = k
        self.selector = None
    
    def fit_transform(self, X, y):
        if self.method == 'none':
            return X
        max_k = min(self.k, X.shape[1])
        self.selector = SelectKBest(score_func=f_classif, k=max_k)
        return self.selector.fit_transform(X, y)
    
    def transform(self, X):
        if self.method == 'none':
            return X
        return self.selector.transform(X)

def save_confusion_matrix(y_true, y_pred, trial_number, dataset_name, phase="Val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'CM - Trial {trial_number} ({dataset_name})')
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.savefig(f'Logs_RF_Baseline_2/{dataset_name}_Trial_{trial_number}_CM.png', bbox_inches='tight')
    plt.close()

datasets_to_test = ['none', 'smote_enn', 'smote_tomek']

for current_dataset in datasets_to_test:
    print(f"Iniciando estudio enfocado en set con oversampling: {current_dataset}")
    
    def objective_rf(trial):
        X_train_raw, y_train_raw = train_datasets[current_dataset]
        total_features = X_train_raw.shape[1]
        
        fs_method = trial.suggest_categorical('feature_selection', ['none', 'f_classif'])
        k_features = trial.suggest_int('k_features', 30, total_features)
        
        selector = FeatureSelector(method=fs_method, k=k_features)
        X_train_fs = selector.fit_transform(X_train_raw, y_train_raw)
        X_val_fs = selector.transform(X_val)
        
        weight_func_name = trial.suggest_categorical('weight_func', ['polynomial', 'effective_samples'])
        if weight_func_name == 'polynomial':
            alpha = trial.suggest_float('poly_alpha', 0.1, 0.9)
            weight_dict = polynomial_class_weight(y_train_raw, alpha=alpha)
        else:
            weight_dict = effective_samples_class_weight(y_train_raw)
        
        rf_params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 400),
            'max_depth': trial.suggest_int('max_depth', 15, 60),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
            'class_weight': weight_dict,
            'n_jobs': -1,
            'random_state': 42
        }
        
        model = RandomForestClassifier(**rf_params)
        model.fit(X_train_fs, y_train_raw)
        y_pred = model.predict(X_val_fs)
        
        f1_mac = f1_score(y_val_1d, y_pred, average='macro')
        report = classification_report(y_val_1d, y_pred, zero_division=0)
        
        save_confusion_matrix(y_val_1d, y_pred, trial.number, current_dataset, phase="Val")
        
        log_msg = f"""Dataset: {current_dataset} | Trial: {trial.number}
FS Method: {fs_method} | K features: {k_features}
Params RF: {trial.params}

F1 Macro: {f1_mac:.4f}
{report}"""
        
        with open(f"Logs_RF_Baseline_2/{current_dataset}_Trial_{trial.number}.log", 'w') as f:
            f.write(log_msg)
            
        return f1_mac

    study_name = f"RF_Opt_{current_dataset}"
    study = optuna.create_study(direction='maximize', study_name=study_name)
    study.optimize(objective_rf, n_trials=30, n_jobs=2)
    
    print(f"\nResultados para {current_dataset.upper()}")
    print(f"Mejor F1 Macro: {study.best_value:.4f}")
    print(f"Mejores Params: {study.best_params}")

[I 2026-03-07 20:35:02,002] A new study created in memory with name: RF_Opt_none


Iniciando estudio enfocado en set con oversampling: none


[I 2026-03-07 20:36:17,122] Trial 1 finished with value: 0.7636614912748293 and parameters: {'feature_selection': 'none', 'k_features': 46, 'weight_func': 'polynomial', 'poly_alpha': 0.14240436471019458, 'n_estimators': 325, 'max_depth': 51, 'min_samples_split': 4, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 1 with value: 0.7636614912748293.
[I 2026-03-07 20:36:18,121] Trial 0 finished with value: 0.7750232840594408 and parameters: {'feature_selection': 'none', 'k_features': 30, 'weight_func': 'polynomial', 'poly_alpha': 0.5815488734701121, 'n_estimators': 385, 'max_depth': 16, 'min_samples_split': 6, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7750232840594408.
[I 2026-03-07 20:36:31,240] Trial 2 finished with value: 0.8102153791370987 and parameters: {'feature_selection': 'none', 'k_features': 54, 'weight_func': 'polynomial', 'poly_alpha': 0.24226726534193777, 'n_estimators': 132, 'max_depth': 40, 'min_samples_split': 8, 'min_sample


Resultados para NONE
Mejor F1 Macro: 0.8414
Mejores Params: {'feature_selection': 'none', 'k_features': 62, 'weight_func': 'effective_samples', 'n_estimators': 168, 'max_depth': 53, 'min_samples_split': 9, 'min_samples_leaf': 2, 'max_features': None}
Iniciando estudio enfocado en set con oversampling: smote_enn


[I 2026-03-07 21:55:46,066] Trial 0 finished with value: 0.7592942762106543 and parameters: {'feature_selection': 'none', 'k_features': 64, 'weight_func': 'polynomial', 'poly_alpha': 0.5532880803142832, 'n_estimators': 144, 'max_depth': 26, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'log2'}. Best is trial 0 with value: 0.7592942762106543.
[I 2026-03-07 21:57:17,355] Trial 2 finished with value: 0.7303595451408157 and parameters: {'feature_selection': 'f_classif', 'k_features': 35, 'weight_func': 'polynomial', 'poly_alpha': 0.5070533306821029, 'n_estimators': 56, 'max_depth': 29, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': None}. Best is trial 0 with value: 0.7592942762106543.
[I 2026-03-07 21:57:17,648] Trial 1 finished with value: 0.816152889457017 and parameters: {'feature_selection': 'f_classif', 'k_features': 48, 'weight_func': 'polynomial', 'poly_alpha': 0.6326619364645399, 'n_estimators': 214, 'max_depth': 43, 'min_samples_split': 4, 'min_sa


Resultados para SMOTE_ENN
Mejor F1 Macro: 0.8442
Mejores Params: {'feature_selection': 'none', 'k_features': 67, 'weight_func': 'effective_samples', 'n_estimators': 337, 'max_depth': 49, 'min_samples_split': 9, 'min_samples_leaf': 2, 'max_features': None}
Iniciando estudio enfocado en set con oversampling: smote_tomek


[I 2026-03-07 23:41:31,741] Trial 0 finished with value: 0.7977510873217355 and parameters: {'feature_selection': 'f_classif', 'k_features': 60, 'weight_func': 'effective_samples', 'n_estimators': 152, 'max_depth': 57, 'min_samples_split': 6, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7977510873217355.
[I 2026-03-07 23:47:53,565] Trial 2 finished with value: 0.8431492712793782 and parameters: {'feature_selection': 'none', 'k_features': 34, 'weight_func': 'effective_samples', 'n_estimators': 51, 'max_depth': 29, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': None}. Best is trial 2 with value: 0.8431492712793782.
[I 2026-03-07 23:49:30,614] Trial 3 finished with value: 0.7845803636114747 and parameters: {'feature_selection': 'none', 'k_features': 36, 'weight_func': 'effective_samples', 'n_estimators': 389, 'max_depth': 32, 'min_samples_split': 9, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 2 with value: 0.8431492712793


Resultados para SMOTE_TOMEK
Mejor F1 Macro: 0.8537
Mejores Params: {'feature_selection': 'none', 'k_features': 38, 'weight_func': 'polynomial', 'poly_alpha': 0.3880771119476566, 'n_estimators': 133, 'max_depth': 25, 'min_samples_split': 4, 'min_samples_leaf': 4, 'max_features': 'sqrt'}


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.base import BaseEstimator, TransformerMixin
import optuna

os.makedirs("Logs_RF_Baseline_3", exist_ok=True)

def save_confusion_matrix(y_true, y_pred, trial_number, phase="Val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - Trial {trial_number} ({phase})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'Logs_RF_Baseline_3/Trial_{trial_number}_RF_{phase}_CM.png', bbox_inches='tight')
    plt.close()

def objective_rf(trial):
    dataset_name = 'smote_tomek'
    X_train_raw, y_train_raw = train_datasets[dataset_name]
    total_features = X_train_raw.shape[1]
    
    fs_method = trial.suggest_categorical('feature_selection', ['none', 'f_classif'])
    k_features = trial.suggest_int('k_features', 50, total_features)
    
    selector = FeatureSelector(method=fs_method, k=k_features) 
    X_train_fs = selector.fit_transform(X_train_raw, y_train_raw)
    X_val_fs = selector.transform(X_val)
    
    weight_dict = effective_samples_class_weight(y_train_raw)
    
    # Acoto el espacio de busqueda para mejorar los resultados
    rf_params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 150),
        'max_depth': trial.suggest_int('max_depth', 25, 45),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 8),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
        'class_weight': weight_dict,
        'n_jobs': -1,
        'random_state': 42
    }
    
    model = RandomForestClassifier(**rf_params)
    model.fit(X_train_fs, y_train_raw)
    y_pred = model.predict(X_val_fs)
    
    f1_mac = f1_score(y_val_1d, y_pred, average='macro')
    report = classification_report(y_val_1d, y_pred, zero_division=0)
    
    save_confusion_matrix(y_val_1d, y_pred, trial.number, phase="Val")
    
    log_msg = f"""Trial {trial.number}
Dataset: {dataset_name} | Feature Selection: {fs_method} (k={k_features})
Params RF: {trial.params}

Metricas Clave
F1 Macro (Validation): {f1_mac:.4f}

Classification Report (Precision, Recall, F1 por clase)
{report}
"""
    
    log_filename = f"Logs_RF_Baseline_3/Trial_{trial.number}_Metrics.log"
    with open(log_filename, 'w', encoding='utf-8') as f:
        f.write(log_msg)
    
    print(f"Trial {trial.number} completado | F1 Macro: {f1_mac:.4f} | Log guardado en {log_filename}")
    
    return f1_mac

print("Iniciando búsqueda enfocada con Optuna para Random Forest...")
study_rf = optuna.create_study(direction='maximize', study_name="RandomForest_IDS_Optimization_Enfocado")

study_rf.optimize(objective_rf, n_trials=100, n_jobs=2) 

print(f" Mejor Trial: {study_rf.best_trial.number}")
print(f" Mejor F1 Macro: {study_rf.best_value:.4f}")
print(f" Mejores Hiperparametros:\n{study_rf.best_params}")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.base import BaseEstimator, TransformerMixin
import optuna

#Tras primeros experimentos me estaba dando recall bajo las class de Web Attack 12,13,14 y quiero probar funciones mas penalizadoras de pesos a ver si esto mejora el F1 Macro
def save_confusion_matrix(y_true, y_pred, trial_number, phase="Val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - Trial {trial_number} ({phase})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'Logs_RF_Baseline_4/Trial_{trial_number}_RF_{phase}_CM.png', bbox_inches='tight')
    plt.close()

def objective_rf(trial):
    dataset_name = trial.suggest_categorical('dataset', ['none', 'smote_enn', 'smote_tomek'])
    
    fs_method = trial.suggest_categorical('feature_selection', ['none', 'f_classif', 'tree'])
    
    X_train_raw, y_train_raw = train_datasets[dataset_name]
    
    selector = FeatureSelector(method=fs_method, k=50) 
    X_train_fs = selector.fit_transform(X_train_raw, y_train_raw)
    X_val_fs = selector.transform(X_val)
    
    weight_func_name = trial.suggest_categorical('weight_func', ['cost_sensitive', 'focal_improved', 'polynomial', 'log_ratio'])
    
    if weight_func_name == 'cost_sensitive':
        weight_dict = cost_sensitive_class_weight(y_train_raw)
        
    elif weight_func_name == 'focal_improved':
        gamma_val = trial.suggest_float('focal_gamma', 1.0, 4.0)
        weight_dict = focal_class_weight_improved(y_train_raw, gamma=gamma_val)
        
    elif weight_func_name == 'polynomial':
        alpha_val = trial.suggest_float('poly_alpha', 0.5, 1.0)
        weight_dict = polynomial_class_weight(y_train_raw, alpha=alpha_val)
        
    elif weight_func_name == 'log_ratio':
        weight_dict = log_ratio_class_weight(y_train_raw)
    
    rf_params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 150),
        'max_depth': trial.suggest_int('max_depth', 15, 50),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 8),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'class_weight': weight_dict,
        'n_jobs': -1,
        'random_state': 42
    }
    
    model = RandomForestClassifier(**rf_params)
    model.fit(X_train_fs, y_train_raw)
    y_pred = model.predict(X_val_fs)
    
    f1_mac = f1_score(y_val_1d, y_pred, average='macro')
    report = classification_report(y_val_1d, y_pred, zero_division=0)
    
    save_confusion_matrix(y_val_1d, y_pred, trial.number, phase="Val")
    
    log_msg = f"""Trial {trial.number}
Dataset: {dataset_name} | Feature Selection: {fs_method}
Params RF: {trial.params}

Metricas Clave
F1 Macro (Validation): {f1_mac:.4f}

Classification Report (Precision, Recall, F1 por clase)
{report}
"""
    
    log_filename = f"Logs_RF_Baseline_4/Trial_{trial.number}_Metrics.log"
    with open(log_filename, 'w', encoding='utf-8') as f:
        f.write(log_msg)
    
    print(f"Trial {trial.number} completado | F1 Macro: {f1_mac:.4f} | Log guardado en {log_filename}")
    
    return f1_mac

print("Iniciando búsqueda con Optuna para Random Forest...")
study_rf = optuna.create_study(direction='maximize', study_name="RandomForest_IDS_Optimization_Heavier_Weights")

study_rf.optimize(objective_rf, n_trials=100, n_jobs=2) 

print(f" Mejor Trial: {study_rf.best_trial.number}")
print(f" Mejor F1 Macro: {study_rf.best_value:.4f}")
print(f" Mejores Hiperparametros:\n{study_rf.best_params}")

[I 2026-03-08 17:48:44,223] A new study created in memory with name: RandomForest_IDS_Optimization_Heavier_Weights


Iniciando búsqueda con Optuna para Random Forest...


[I 2026-03-08 17:49:04,551] Trial 1 finished with value: 0.7662304600879213 and parameters: {'dataset': 'smote_enn', 'feature_selection': 'none', 'weight_func': 'polynomial', 'poly_alpha': 0.8263260689137697, 'n_estimators': 135, 'max_depth': 37, 'min_samples_split': 3, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 1 with value: 0.7662304600879213.


Trial 1 completado | F1 Macro: 0.7662 | Log guardado en Logs_RF_Baseline_4/Trial_1_Metrics.log


[I 2026-03-08 17:49:20,547] Trial 0 finished with value: 0.7800258996610289 and parameters: {'dataset': 'smote_tomek', 'feature_selection': 'f_classif', 'weight_func': 'cost_sensitive', 'n_estimators': 136, 'max_depth': 48, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7800258996610289.


...

Trial 97 completado | F1 Macro: 0.8568 | Log guardado en Logs_RF_Baseline_4/Trial_97_Metrics.log


[I 2026-03-08 18:16:35,501] Trial 98 finished with value: 0.8547512145121053 and parameters: {'dataset': 'smote_tomek', 'feature_selection': 'tree', 'weight_func': 'log_ratio', 'n_estimators': 114, 'max_depth': 26, 'min_samples_split': 6, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 19 with value: 0.8588820449981469.


Trial 98 completado | F1 Macro: 0.8548 | Log guardado en Logs_RF_Baseline_4/Trial_98_Metrics.log


[I 2026-03-08 18:16:44,221] Trial 99 finished with value: 0.7518404948572522 and parameters: {'dataset': 'smote_enn', 'feature_selection': 'tree', 'weight_func': 'log_ratio', 'n_estimators': 101, 'max_depth': 26, 'min_samples_split': 6, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 19 with value: 0.8588820449981469.


Trial 99 completado | F1 Macro: 0.7518 | Log guardado en Logs_RF_Baseline_4/Trial_99_Metrics.log
 Mejor Trial: 19
 Mejor F1 Macro: 0.8589
 Mejores Hiperparametros:
{'dataset': 'smote_tomek', 'feature_selection': 'tree', 'weight_func': 'log_ratio', 'n_estimators': 150, 'max_depth': 30, 'min_samples_split': 7, 'min_samples_leaf': 2, 'max_features': 'sqrt'}


In [22]:
#Tests para intentar mejorar la busqueda de intrusiones web
def save_confusion_matrix_new(testSet, y_true, y_pred, trial_number, phase="Val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - Trial {trial_number} ({phase})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'Logs_RF_Baseline_{testSet}/Trial_{trial_number}_RF_{phase}_CM.png', bbox_inches='tight')
    plt.close()

def objective_rf_new(trial):
    dataset_name = 'smote_tomek'
    
    fs_method = trial.suggest_categorical('feature_selection', ['none', 'f_classif', 'tree'])
    X_train_raw, y_train_raw = train_datasets[dataset_name]
    
    total_cols = X_train_raw.shape[1]
    k_feats = trial.suggest_int('k_features', 30, total_cols)
    
    selector = FeatureSelector(method=fs_method, k=k_feats) 
    X_train_fs = selector.fit_transform(X_train_raw, y_train_raw)
    X_val_fs = selector.transform(X_val)
    
    weight_func_name = trial.suggest_categorical('weight_func', ['log_ratio', 'focal_improved','effective_samples_class_weight'])
    
    if weight_func_name == 'focal_improved':
        gamma_val = trial.suggest_float('focal_gamma', 2.0, 4.0)
        weight_dict = focal_class_weight_improved(y_train_raw, gamma=gamma_val)
    elif weight_func_name == 'log_ratio':
        weight_dict = log_ratio_class_weight(y_train_raw)
    elif weight_func_name == 'effective_samples_class_weight':
        weight_dict = effective_samples_class_weight(y_train_raw)
        
    web_boost = trial.suggest_float('web_boost', 1.0, 5.0) 
    bot_boost = trial.suggest_float('bot_boost', 0.5, 3.0) 
    
    if 12 in weight_dict: weight_dict[12] *= web_boost
    if 13 in weight_dict: weight_dict[13] *= (web_boost * 2)
    if 14 in weight_dict: weight_dict[14] *= web_boost
    if 1 in weight_dict: weight_dict[1] *= bot_boost
    
    rf_params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'max_depth': trial.suggest_int('max_depth', 25, 50),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 8),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 4),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'class_weight': weight_dict,
        'n_jobs': -1,
        'random_state': 42
    }
    
    model = RandomForestClassifier(**rf_params)
    model.fit(X_train_fs, y_train_raw)
    y_pred = model.predict(X_val_fs)
    
    f1_mac = f1_score(y_val_1d, y_pred, average='macro')
    report = classification_report(y_val_1d, y_pred, zero_division=0)
    
    save_confusion_matrix_new(5, y_val_1d, y_pred, trial.number, phase="Val")
    
    log_msg = f"""Trial {trial.number}
Dataset: {dataset_name} | Feature Selection: {fs_method} (k={k_feats})
Params RF: {trial.params}

Metricas Clave
F1 Macro (Validation): {f1_mac:.4f}

Classification Report (Precision, Recall, F1 por clase)
{report}
"""
    
    log_filename = f"Logs_RF_Baseline_5/Trial_{trial.number}_Metrics.log"
    with open(log_filename, 'w', encoding='utf-8') as f:
        f.write(log_msg)
    
    print(f"Trial {trial.number} completado | F1 Macro: {f1_mac:.4f} | Log guardado en {log_filename}")
    
    return f1_mac

print("Iniciando búsqueda con Optuna para Random Forest...")
study_rf_new = optuna.create_study(direction='maximize', study_name="RandomForest_IDS_Optimization_Specialized_Attacks")

study_rf_new.optimize(objective_rf_new, n_trials=200, n_jobs=2) 

print(f" Mejor Trial: {study_rf_new.best_trial.number}")
print(f" Mejor F1 Macro: {study_rf_new.best_value:.4f}")
print(f" Mejores Hiperparametros:\n{study_rf_new.best_params}")

[I 2026-03-08 18:33:30,543] A new study created in memory with name: RandomForest_IDS_Optimization_Specialized_Attacks


Iniciando búsqueda con Optuna para Random Forest...


[I 2026-03-08 18:34:09,755] Trial 1 finished with value: 0.7630547236256928 and parameters: {'feature_selection': 'f_classif', 'k_features': 43, 'weight_func': 'effective_samples_class_weight', 'web_boost': 2.1489478072143724, 'bot_boost': 2.128043123432512, 'n_estimators': 195, 'max_depth': 45, 'min_samples_split': 4, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.7630547236256928.


Trial 1 completado | F1 Macro: 0.7631 | Log guardado en Logs_RF_Baseline_5/Trial_1_Metrics.log


[I 2026-03-08 18:34:26,768] Trial 0 finished with value: 0.7879616017790312 and parameters: {'feature_selection': 'tree', 'k_features': 57, 'weight_func': 'log_ratio', 'web_boost': 3.664715878064907, 'bot_boost': 1.2993562479709435, 'n_estimators': 269, 'max_depth': 27, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7879616017790312.


...

Trial 197 completado | F1 Macro: 0.7941 | Log guardado en Logs_RF_Baseline_5/Trial_197_Metrics.log


[I 2026-03-08 19:57:36,835] Trial 198 finished with value: 0.7931936755066424 and parameters: {'feature_selection': 'tree', 'k_features': 35, 'weight_func': 'log_ratio', 'web_boost': 1.2340242169962123, 'bot_boost': 2.885861722351828, 'n_estimators': 263, 'max_depth': 46, 'min_samples_split': 8, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 165 with value: 0.8619061149251235.


Trial 198 completado | F1 Macro: 0.7932 | Log guardado en Logs_RF_Baseline_5/Trial_198_Metrics.log


[I 2026-03-08 19:57:47,166] Trial 199 finished with value: 0.8599012662503864 and parameters: {'feature_selection': 'tree', 'k_features': 37, 'weight_func': 'log_ratio', 'web_boost': 1.2514014609709645, 'bot_boost': 2.9598143786128808, 'n_estimators': 261, 'max_depth': 44, 'min_samples_split': 8, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 165 with value: 0.8619061149251235.


Trial 199 completado | F1 Macro: 0.8599 | Log guardado en Logs_RF_Baseline_5/Trial_199_Metrics.log
 Mejor Trial: 165
 Mejor F1 Macro: 0.8619
 Mejores Hiperparametros:
{'feature_selection': 'tree', 'k_features': 36, 'weight_func': 'log_ratio', 'web_boost': 2.000974185366484, 'bot_boost': 2.939178439206625, 'n_estimators': 279, 'max_depth': 43, 'min_samples_split': 8, 'min_samples_leaf': 2, 'max_features': 'sqrt'}


In [13]:
def save_confusion_matrix_new(testset, y_true, y_pred, trial_number, dataset_name, phase="val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion matrix - {dataset_name} trial {trial_number} ({phase})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'Logs_RF_Baseline_{testset}/{dataset_name}_trial_{trial_number}_rf_{phase}_cm.png', bbox_inches='tight')
    plt.close()

In [ ]:
import os
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

log_folder = "Logs_RF_Baseline_6"
os.makedirs(log_folder, exist_ok=True)

datasets_to_test = ['none', 'smote', 'smote_enn', 'smote_tomek']
resultados_globales = {}

for current_dataset in datasets_to_test:
    print(f"Iniciando estudio enfocado en set: {current_dataset}")
    
    def objective_rf(trial):
        x_train_raw, y_train_raw = train_datasets_2[current_dataset]
        total_features = x_train_raw.shape[1]

        weight_func_name = trial.suggest_categorical('weight_func', ['none', 'log_ratio', 'cost_sensitive', 'focal_improved'])
        
        if weight_func_name == 'none':
            weight_dict = None
        else:
            if weight_func_name == 'focal_improved':
                gamma_val = trial.suggest_float('focal_gamma', 2.0, 4.0)
                weight_dict = focal_class_weight_improved(y_train_raw, gamma=gamma_val)
            elif weight_func_name == 'log_ratio':
                weight_dict = log_ratio_class_weight(y_train_raw)
            elif weight_func_name == 'cost_sensitive':
                weight_dict = cost_sensitive_class_weight(y_train_raw)

            web_boost = trial.suggest_float('web_boost', 1.0, 3.0) 
            bot_boost = trial.suggest_float('bot_boost', 1.5, 3.5) 
            
            if 12 in weight_dict: weight_dict[12] *= web_boost
            if 13 in weight_dict: weight_dict[13] *= (web_boost * 2)
            if 14 in weight_dict: weight_dict[14] *= web_boost
            if 1 in weight_dict: weight_dict[1] *= bot_boost

        fs_method = trial.suggest_categorical('fs_method', ['none', 'tree'])
        
        if fs_method == 'none':
            k_features = total_features
        else:
            k_features = trial.suggest_int('k_features', 25, total_features)
        
        selector = FeatureSelector(method=fs_method, k=k_features, class_weight=weight_dict) 
        x_train_fs = selector.fit_transform(x_train_raw, y_train_raw)
        x_val_fs = selector.transform(X_val)
        
        rf_params = {
            'n_estimators': trial.suggest_int('n_estimators', 200, 350),
            'max_depth': trial.suggest_int('max_depth', 35, 50),
            'min_samples_split': trial.suggest_int('min_samples_split', 4, 10),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 3),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
            'class_weight': weight_dict,
            'n_jobs': -1,
            'random_state': 42
        }
        
        model = RandomForestClassifier(**rf_params)
        model.fit(x_train_fs, y_train_raw)
        y_pred = model.predict(x_val_fs)
        
        f1_mac = f1_score(y_val_1d, y_pred, average='macro')
        report = classification_report(y_val_1d, y_pred, zero_division=0)
        
        save_confusion_matrix_new(6, y_val_1d, y_pred, trial.number, current_dataset, phase="val")
        
        log_msg = f"""Dataset: {current_dataset} | trial: {trial.number}
fs method: {fs_method} | k features: {k_features}
params rf: {trial.params}

F1 macro: {f1_mac:.4f}
{report}"""
        
        with open(f"{log_folder}/{current_dataset}_trial_{trial.number}.log", 'w') as f:
            f.write(log_msg)
            
        return f1_mac

    study_name = f"rf_opt_{current_dataset}"
    study = optuna.create_study(direction='maximize', study_name=study_name)
    study.optimize(objective_rf, n_trials=75, n_jobs=1)
    
    resultados_globales[current_dataset] = {
        'best_trial': study.best_trial.number,
        'best_f1_macro': study.best_value,
        'best_params': study.best_params
    }
    
    best_log_msg = f"""Mejor trial para dataset: {current_dataset}
Trial numero: {study.best_trial.number}
F1 macro alcanzado: {study.best_value:.4f}
Hiperparametros:
{study.best_params}
"""
    with open(f"{log_folder}/mejor_trial_{current_dataset}.log", 'w') as f:
        f.write(best_log_msg)

    print(f"\nResultados finales para {current_dataset}")
    print(f"Mejor trial: {study.best_trial.number}")
    print(f"Mejor F1 macro: {study.best_value:.4f}")
    print(f"Mejores params: {study.best_params}\n")

informe_final = "Informe final: Competencia de datasets con y sin Oversampling\n"

for dataset, metricas in resultados_globales.items():
    informe_final += f"Dataset: {dataset}\n"
    informe_final += f"- Mejor trial: {metricas['best_trial']}\n"
    informe_final += f"- F1 macro alcanzado: {metricas['best_f1_macro']:.4f}\n"
    informe_final += f"- Hiperparametros ganadores:\n"
    for param, value in metricas['best_params'].items():
        informe_final += f"* {param}: {value}\n"

ganador_absoluto = max(resultados_globales, key=lambda x: resultados_globales[x]['best_f1_macro'])
informe_final += f"\nGanador Absoluto: {ganador_absoluto} (F1 macro: {resultados_globales[ganador_absoluto]['best_f1_macro']:.4f})\n"

print(informe_final)

with open(f"{log_folder}/resumen_global_resultados.txt", 'w') as f:
    f.write(informe_final)

print(f"El informe final ha sido guardado en {log_folder}/resumen_global_resultados.txt")

[I 2026-03-08 23:43:17,996] A new study created in memory with name: rf_opt_none


Iniciando estudio enfocado en set: none


[I 2026-03-08 23:43:49,786] Trial 0 finished with value: 0.7639941191140344 and parameters: {'weight_func': 'log_ratio', 'web_boost': 1.6160020402134583, 'bot_boost': 2.904387529914583, 'fs_method': 'none', 'n_estimators': 337, 'max_depth': 43, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 0 with value: 0.7639941191140344.
[I 2026-03-08 23:44:19,159] Trial 1 finished with value: 0.7951780470279977 and parameters: {'weight_func': 'log_ratio', 'web_boost': 2.048217925290224, 'bot_boost': 2.720310240061136, 'fs_method': 'none', 'n_estimators': 311, 'max_depth': 43, 'min_samples_split': 8, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 1 with value: 0.7951780470279977.
[I 2026-03-08 23:45:04,177] Trial 2 finished with value: 0.7609108564027776 and parameters: {'weight_func': 'none', 'fs_method': 'tree', 'k_features': 52, 'n_estimators': 265, 'max_depth': 49, 'min_samples_split': 9, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best i


Resultados finales para none
Mejor trial: 63
Mejor F1 macro: 0.8533
Mejores params: {'weight_func': 'cost_sensitive', 'web_boost': 1.731189591019194, 'bot_boost': 1.5822756572201808, 'fs_method': 'tree', 'k_features': 31, 'n_estimators': 275, 'max_depth': 41, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2'}

Iniciando estudio enfocado en set: smote


[I 2026-03-09 00:24:53,360] Trial 0 finished with value: 0.8057238767599817 and parameters: {'weight_func': 'log_ratio', 'web_boost': 1.7311418719898406, 'bot_boost': 1.6590285380924088, 'fs_method': 'none', 'n_estimators': 304, 'max_depth': 42, 'min_samples_split': 6, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 0 with value: 0.8057238767599817.
[I 2026-03-09 00:25:30,002] Trial 1 finished with value: 0.8415316378844557 and parameters: {'weight_func': 'focal_improved', 'focal_gamma': 2.824543107519078, 'web_boost': 1.4808925074347687, 'bot_boost': 2.1169769927054567, 'fs_method': 'tree', 'k_features': 30, 'n_estimators': 215, 'max_depth': 44, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.8415316378844557.
[I 2026-03-09 00:26:17,223] Trial 2 finished with value: 0.820262894203617 and parameters: {'weight_func': 'cost_sensitive', 'web_boost': 2.8905327344469365, 'bot_boost': 2.5381402997176252, 'fs_method': 'tree',


Resultados finales para smote
Mejor trial: 48
Mejor F1 macro: 0.8789
Mejores params: {'weight_func': 'none', 'fs_method': 'tree', 'k_features': 32, 'n_estimators': 226, 'max_depth': 42, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'sqrt'}

Iniciando estudio enfocado en set: smote_enn


[I 2026-03-09 01:15:23,151] Trial 0 finished with value: 0.7921218924000475 and parameters: {'weight_func': 'cost_sensitive', 'web_boost': 2.06376445033981, 'bot_boost': 1.8431389663347983, 'fs_method': 'none', 'n_estimators': 295, 'max_depth': 35, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7921218924000475.
[I 2026-03-09 01:15:55,105] Trial 1 finished with value: 0.792437657834159 and parameters: {'weight_func': 'none', 'fs_method': 'none', 'n_estimators': 268, 'max_depth': 41, 'min_samples_split': 10, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 1 with value: 0.792437657834159.
[I 2026-03-09 01:16:36,366] Trial 2 finished with value: 0.7940740651011603 and parameters: {'weight_func': 'focal_improved', 'focal_gamma': 3.1585879466119415, 'web_boost': 2.0727442388127613, 'bot_boost': 2.7923437693919775, 'fs_method': 'none', 'n_estimators': 327, 'max_depth': 45, 'min_samples_split': 9, 'min_samples_leaf': 2, 'ma


Resultados finales para smote_enn
Mejor trial: 62
Mejor F1 macro: 0.8689
Mejores params: {'weight_func': 'log_ratio', 'web_boost': 2.621853986343816, 'bot_boost': 2.8234200656467965, 'fs_method': 'tree', 'k_features': 35, 'n_estimators': 335, 'max_depth': 36, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 'log2'}

Iniciando estudio enfocado en set: smote_tomek


[I 2026-03-09 02:05:15,610] Trial 0 finished with value: 0.8200637737944979 and parameters: {'weight_func': 'none', 'fs_method': 'tree', 'k_features': 48, 'n_estimators': 212, 'max_depth': 40, 'min_samples_split': 8, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 0 with value: 0.8200637737944979.
[I 2026-03-09 02:05:53,844] Trial 1 finished with value: 0.8269076929675119 and parameters: {'weight_func': 'none', 'fs_method': 'none', 'n_estimators': 295, 'max_depth': 42, 'min_samples_split': 9, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 1 with value: 0.8269076929675119.
[I 2026-03-09 02:06:35,290] Trial 2 finished with value: 0.8237343540332701 and parameters: {'weight_func': 'focal_improved', 'focal_gamma': 2.9369869916936944, 'web_boost': 1.1785823148353274, 'bot_boost': 1.9147812832249225, 'fs_method': 'tree', 'k_features': 29, 'n_estimators': 254, 'max_depth': 43, 'min_samples_split': 9, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 1 


Resultados finales para smote_tomek
Mejor trial: 8
Mejor F1 macro: 0.8303
Mejores params: {'weight_func': 'none', 'fs_method': 'tree', 'k_features': 61, 'n_estimators': 310, 'max_depth': 49, 'min_samples_split': 4, 'min_samples_leaf': 3, 'max_features': 'sqrt'}

Informe final: Competencia de datasets con y sin Oversampling
Dataset: none
- Mejor trial: 63
- F1 macro alcanzado: 0.8533
- Hiperparametros ganadores:
* weight_func: cost_sensitive
* web_boost: 1.731189591019194
* bot_boost: 1.5822756572201808
* fs_method: tree
* k_features: 31
* n_estimators: 275
* max_depth: 41
* min_samples_split: 5
* min_samples_leaf: 2
* max_features: log2
Dataset: smote
- Mejor trial: 48
- F1 macro alcanzado: 0.8789
- Hiperparametros ganadores:
* weight_func: none
* fs_method: tree
* k_features: 32
* n_estimators: 226
* max_depth: 42
* min_samples_split: 10
* min_samples_leaf: 1
* max_features: sqrt
Dataset: smote_enn
- Mejor trial: 62
- F1 macro alcanzado: 0.8689
- Hiperparametros ganadores:
* weight_f

In [ ]:
print("Prueba final en test set")

best_params = study_rf.best_params
best_dataset = best_params['dataset']
best_fs = best_params['feature_selection']
X_train_best, y_train_best = train_datasets[best_dataset]

print(f"Aplicando Feature Selection: {best_fs}...")
final_selector = FeatureSelector(method=best_fs, k=50)
X_train_final = final_selector.fit_transform(X_train_best, y_train_best)
X_test_final = final_selector.transform(X_test)

if best_params['weight_func'] == 'polynomial':
    final_weights = polynomial_class_weight(y_train_best, alpha=best_params['poly_alpha'])
else:
    final_weights = effective_samples_class_weight(y_train_best)

print("Entrenando Random Forest con mejores hiperparámetros...")
final_rf = RandomForestClassifier(
    n_estimators=best_params['n_estimators'],
    max_depth=best_params['max_depth'],
    min_samples_split=best_params['min_samples_split'],
    class_weight=final_weights,
    n_jobs=-1,
    random_state=42
)
final_rf.fit(X_train_final, y_train_best)

y_pred_test = final_rf.predict(X_test_final)

test_f1_mac = f1_score(y_test_1d, y_pred_test, average='macro')
test_report = classification_report(y_test_1d, y_pred_test)

print(f"Resultados Finales en Test Set")
print(f"F1 Macro: {test_f1_mac:.4f}")
print("\nReporte de Clasificación (Test):")
print(test_report)

save_confusion_matrix(y_test_1d, y_pred_test, study_rf.best_trial.number, phase="Test_Final")

logger.info(f"""
Evaluacion Final En Test
Best Trial: {study_rf.best_trial.number}
Dataset: {best_dataset} | Feature Selection: {best_fs}
F1 Macro (Test): {test_f1_mac:.4f}

Metricas por class (Test):
{test_report}
""")
print("\nPipeline de Random Forest completado. Logs y matrices guardados en 'Logs_RF_Baseline/'.")

Prueba final en test set
Aplicando Feature Selection: f_classif...


Entrenando Random Forest con mejores hiperparámetros...
Resultados Finales en Test Set
F1 Macro: 0.8612

Reporte de Clasificación (Test):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    307072
           1       0.48      0.67      0.56       295
           2       1.00      1.00      1.00     19204
           3       1.00      0.99      1.00      1544
           4       1.00      1.00      1.00     34661
           5       0.99      0.99      0.99       825
           6       1.00      0.99      0.99       870
           7       1.00      1.00      1.00      1190
           8       1.00      1.00      1.00         2
           9       1.00      0.80      0.89         5
          10       0.99      1.00      0.99     23840
          11       0.99      1.00      0.99       884
          12       0.75      0.77      0.76       226
          13       0.33      0.33      0.33         3
          14       0.41      0.42      0.41        

In [14]:
def save_confusion_matrix(y_true, y_pred, trial_number, phase="Val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - Trial {trial_number} ({phase})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f'Logs_RF_Baseline_3/Trial_{trial_number}_RF_{phase}_CM.png', bbox_inches='tight')
    plt.close()

print("Generando segunda prueba final en test set")

best_dataset = 'smote_tomek'
best_fs = 'f_classif'
best_k = 50

best_rf_params = {
    'n_estimators': 102,
    'max_depth': 40,
    'min_samples_split': 7,
    'min_samples_leaf': 5,
    'max_features': 'sqrt',
    'criterion': 'gini'
}

X_train_best, y_train_best = train_datasets[best_dataset]

print(f"Aplicando Feature Selection: {best_fs} con k={best_k}...")
final_selector = FeatureSelector(method=best_fs, k=best_k)
X_train_final = final_selector.fit_transform(X_train_best, y_train_best)
X_test_final = final_selector.transform(X_test)

print("Calculando pesos de clases (effective_samples)...")
final_weights = effective_samples_class_weight(y_train_best)

print("Entrenando Random Forest final...")
final_rf = RandomForestClassifier(
    **best_rf_params,
    class_weight=final_weights,
    n_jobs=-1,
    random_state=42
)
final_rf.fit(X_train_final, y_train_best)

print("Evaluando predicciones en el Test Set...")
y_pred_test = final_rf.predict(X_test_final)

test_f1_mac = f1_score(y_test_1d, y_pred_test, average='macro')
test_report = classification_report(y_test_1d, y_pred_test, zero_division=0)

print(f"\nResultados Finales en Test Set")
print(f"F1 Macro: {test_f1_mac:.4f}")
print("\nReporte de Clasificación (Test):")
print(test_report)

save_confusion_matrix(y_test_1d, y_pred_test, trial_number="Final", phase="Test")

log_msg = f"""Evaluacion Final con Test
Modelo Base: Random Forest
Dataset: {best_dataset} | Feature Selection: {best_fs} (k={best_k})
Params RF: {best_rf_params}

F1 Macro (Test): {test_f1_mac:.4f}

Metricas por clase (Test):
{test_report}
"""

log_filename = "Logs_RF_Baseline_3/RF_FINAL_TEST_Metrics.log"
with open(log_filename, 'w', encoding='utf-8') as f:
    f.write(log_msg)

print(f"\nPipeline de Random Forest completado con éxito.")
print(f"Resultados guardados en: {log_filename}")

Generando segunda prueba final en test set
Aplicando Feature Selection: f_classif con k=50...
Calculando pesos de clases (effective_samples)...
Entrenando Random Forest final...
Evaluando predicciones en el Test Set...

Resultados Finales en Test Set
F1 Macro: 0.8363

Reporte de Clasificación (Test):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    307072
           1       0.55      0.65      0.60       295
           2       1.00      1.00      1.00     19204
           3       0.99      0.99      0.99      1544
           4       1.00      1.00      1.00     34661
           5       0.99      0.99      0.99       825
           6       1.00      0.99      1.00       870
           7       1.00      1.00      1.00      1190
           8       1.00      1.00      1.00         2
           9       1.00      0.80      0.89         5
          10       0.99      1.00      1.00     23840
          11       0.98      1.00      0.99      

In [20]:
print("Prueba final en test set con el mejor modelo")

best_dataset = 'smote'
best_trial = 48
best_fs = 'tree'
best_k = 32
x_train_best, y_train_best = train_datasets_2[best_dataset]

print(f"aplicando feature selection: {best_fs} con k={best_k}...")
final_selector = FeatureSelector(method=best_fs, k=best_k, class_weight=None)
x_train_final = final_selector.fit_transform(x_train_best, y_train_best)
x_test_final = final_selector.transform(X_test) 

best_rf_params = {
    'n_estimators': 226,
    'max_depth': 42,
    'min_samples_split': 10,
    'min_samples_leaf': 1,
    'max_features': 'sqrt',
    'class_weight': None,
    'n_jobs': -1,
    'random_state': 42
}

print("entrenando random forest con los mejores hiperparametros...")
final_rf = RandomForestClassifier(**best_rf_params)
final_rf.fit(x_train_final, y_train_best)

print("evaluando en el set de test...")
y_pred_test = final_rf.predict(x_test_final)

test_f1_mac = f1_score(y_test_1d, y_pred_test, average='macro')
test_report = classification_report(y_test_1d, y_pred_test, zero_division=0)

print("resultados finales en test set")
print(f"f1 macro: {test_f1_mac:.4f}")
print("\nreporte de clasificacion (test):")
print(test_report)

save_confusion_matrix_new(
    testset="6", 
    y_true=y_test_1d, 
    y_pred=y_pred_test, 
    trial_number=best_trial, 
    dataset_name=best_dataset, 
    phase="test_final"
)

log_msg = f"""Evaluacion final con test
modelo base: random forest
dataset: {best_dataset} | feature selection: {best_fs} (k={best_k})
params rf: {best_rf_params}

f1 macro (test): {test_f1_mac:.4f}

metricas por clase (test):
{test_report}
"""

log_filename = "Logs_RF_Baseline_6/rf_final_test_metrics.log"
with open(log_filename, 'w', encoding='utf-8') as f:
    f.write(log_msg)

print("\npipeline de random forest completado con exito.")
print(f"resultados guardados en: {log_filename}")

Prueba final en test set con el mejor modelo
aplicando feature selection: tree con k=32...


entrenando random forest con los mejores hiperparametros...
evaluando en el set de test...
resultados finales en test set
f1 macro: 0.8294

reporte de clasificacion (test):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    307072
           1       0.45      0.71      0.55       295
           2       1.00      1.00      1.00     19204
           3       1.00      1.00      1.00      1544
           4       1.00      1.00      1.00     34661
           5       0.99      0.99      0.99       825
           6       1.00      0.99      0.99       870
           7       1.00      1.00      1.00      1190
           8       1.00      1.00      1.00         2
           9       1.00      0.80      0.89         5
          10       0.99      1.00      0.99     23840
          11       0.98      1.00      0.99       884
          12       0.69      0.73      0.71       226
          13       0.00      0.00      0.00         3
          14    

In [9]:
import joblib
X_train_none_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_none.joblib")
y_train_none_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_none.joblib")

X_train_smote_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_smote.joblib")
y_train_smote_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_smote.joblib")

X_train_smote_enn_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_smote_enn.joblib")
y_train_smote_enn_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_smote_enn.joblib")

X_train_smote_tomek_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_smote_tomek.joblib")
y_train_smote_tomek_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_smote_tomek.joblib")

X_val_imp_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/X_val_scaled_bayesian_grouped.pkl")
X_test_imp_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/X_test_scaled_bayesian_grouped.pkl")

y_val_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/y_val_grouped.pkl")
y_test_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/y_test_grouped.pkl")

In [6]:
train_datasets_grouped = {
    'none': (X_train_none_grouped, y_train_none_grouped.values.ravel() if isinstance(y_train_none_grouped, pd.DataFrame) else y_train_none_grouped.ravel()),
    'smote': (X_train_smote_grouped, y_train_smote_grouped.values.ravel() if isinstance(y_train_smote_grouped, pd.DataFrame) else y_train_smote_grouped.ravel()),
    'smote_enn': (X_train_smote_enn_grouped, y_train_smote_enn_grouped.values.ravel() if isinstance(y_train_smote_enn_grouped, pd.DataFrame) else y_train_smote_enn_grouped.ravel()),
    'smote_tomek': (X_train_smote_tomek_grouped, y_train_smote_tomek_grouped.values.ravel() if isinstance(y_train_smote_tomek_grouped, pd.DataFrame) else y_train_smote_tomek_grouped.ravel())
}

In [ ]:
import os
import optuna
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

log_folder = "Logs_RF_Baseline_7"
os.makedirs(log_folder, exist_ok=True)

datasets_to_test = ['none', 'smote', 'smote_enn', 'smote_tomek']
resultados_globales = {}

for current_dataset in datasets_to_test:
    print(f"\nIniciando estudio enfocado en set agrupado: {current_dataset}")
    
    def objective_rf(trial):
        x_train_raw, y_train_raw = train_datasets_grouped[current_dataset]
        total_features = x_train_raw.shape[1]

        weight_func_name = trial.suggest_categorical('weight_func', ['none', 'balanced', 'log_ratio', 'cost_sensitive'])
        
        if weight_func_name == 'none':
            weight_dict = None
        else:
            if weight_func_name == 'balanced':
                classes_in_data = np.unique(y_train_raw)
                weights_arr = compute_class_weight('balanced', classes=classes_in_data, y=y_train_raw)
                weight_dict = dict(zip(classes_in_data, weights_arr))
            elif weight_func_name == 'log_ratio':
                weight_dict = log_ratio_class_weight(y_train_raw)
            elif weight_func_name == 'cost_sensitive':
                weight_dict = cost_sensitive_class_weight(y_train_raw)

            web_boost = trial.suggest_float('web_boost', 1.0, 3.0) 
            bot_boost = trial.suggest_float('bot_boost', 1.5, 3.5) 
            
            if 12 in weight_dict: weight_dict[12] *= web_boost
            if 1 in weight_dict: weight_dict[1] *= bot_boost

        fs_method = trial.suggest_categorical('fs_method', ['none', 'tree'])
        
        if fs_method == 'none':
            k_features = total_features
        else:
            k_features = trial.suggest_int('k_features', 20, total_features)
        
        selector = FeatureSelector(method=fs_method, k=k_features, class_weight=weight_dict) 
        x_train_fs = selector.fit_transform(x_train_raw, y_train_raw)
        x_val_fs = selector.transform(X_val_imp_grouped)
        
        rf_params = {
            'n_estimators': trial.suggest_int('n_estimators', 200, 350),
            'max_depth': trial.suggest_int('max_depth', 35, 50),
            'min_samples_split': trial.suggest_int('min_samples_split', 4, 10),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 3),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
            'class_weight': weight_dict,
            'n_jobs': -1,
            'random_state': 42
        }
        
        model = RandomForestClassifier(**rf_params)
        model.fit(x_train_fs, y_train_raw)
        y_pred = model.predict(x_val_fs)
        
        f1_mac = f1_score(y_val_grouped, y_pred, average='macro')
        report = classification_report(y_val_grouped, y_pred, zero_division=0)
        
        save_confusion_matrix_new(7, y_val_grouped, y_pred, trial.number, current_dataset, phase="val")
        
        log_msg = f"""dataset: {current_dataset} | trial: {trial.number}
fs method: {fs_method} | k features: {k_features}
params rf: {trial.params}

F1 macro: {f1_mac:.4f}
{report}"""
        
        with open(f"{log_folder}/{current_dataset}_trial_{trial.number}.log", 'w') as f:
            f.write(log_msg)
            
        return f1_mac

    study_name = f"rf_opt_grouped_{current_dataset}"
    study = optuna.create_study(direction='maximize', study_name=study_name)
    study.optimize(objective_rf, n_trials=75, n_jobs=1)
    
    resultados_globales[current_dataset] = {
        'best_trial': study.best_trial.number,
        'best_f1_macro': study.best_value,
        'best_params': study.best_params
    }
    
    best_log_msg = f"""Mejor trial para dataset agrupado: {current_dataset}
trial numero: {study.best_trial.number}
F1 macro alcanzado: {study.best_value:.4f}
hiperparametros:
{study.best_params}
"""
    with open(f"{log_folder}/mejor_trial_{current_dataset}.log", 'w') as f:
        f.write(best_log_msg)

    print(f"\nResultados finales para {current_dataset}")
    print(f"Mejor trial: {study.best_trial.number}")
    print(f"Mejor F1 macro: {study.best_value:.4f}")
    print(f"Mejores params: {study.best_params}\n")

informe_final = "Informe final: competencia de datasets agrupados\n"

for dataset, metricas in resultados_globales.items():
    informe_final += f"dataset: {dataset}\n"
    informe_final += f"- mejor trial: {metricas['best_trial']}\n"
    informe_final += f"- f1 macro alcanzado: {metricas['best_f1_macro']:.4f}\n"
    informe_final += f"- hiperparametros ganadores:\n"
    for param, value in metricas['best_params'].items():
        informe_final += f"* {param}: {value}\n"

ganador_absoluto = max(resultados_globales, key=lambda x: resultados_globales[x]['best_f1_macro'])
informe_final += f"\nganador absoluto: {ganador_absoluto} (f1 macro: {resultados_globales[ganador_absoluto]['best_f1_macro']:.4f})\n"

print(informe_final)

with open(f"{log_folder}/resumen_global_resultados.txt", 'w') as f:
    f.write(informe_final)

print(f"El informe final ha sido guardado en {log_folder}/resumen_global_resultados.txt")

[I 2026-03-09 04:39:32,237] A new study created in memory with name: rf_opt_grouped_none



Iniciando estudio enfocado en set agrupado: none


[I 2026-03-09 04:40:04,531] Trial 0 finished with value: 0.9567380237199669 and parameters: {'weight_func': 'balanced', 'web_boost': 2.104147111723589, 'bot_boost': 1.835980010006102, 'fs_method': 'none', 'n_estimators': 315, 'max_depth': 35, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 0 with value: 0.9567380237199669.
[I 2026-03-09 04:40:32,668] Trial 1 finished with value: 0.9420983543186494 and parameters: {'weight_func': 'log_ratio', 'web_boost': 1.2898614264595267, 'bot_boost': 3.323224486816859, 'fs_method': 'tree', 'k_features': 35, 'n_estimators': 211, 'max_depth': 35, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 0 with value: 0.9567380237199669.
[I 2026-03-09 04:41:10,033] Trial 2 finished with value: 0.9504388423891265 and parameters: {'weight_func': 'balanced', 'web_boost': 2.6629936145586663, 'bot_boost': 1.6845904897724606, 'fs_method': 'tree', 'k_features': 34, 'n_estimators': 311, 'max_dep


Resultados finales para none
Mejor trial: 25
Mejor F1 macro: 0.9656
Mejores params: {'weight_func': 'none', 'fs_method': 'tree', 'k_features': 57, 'n_estimators': 289, 'max_depth': 37, 'min_samples_split': 4, 'min_samples_leaf': 2, 'max_features': 'sqrt'}


Iniciando estudio enfocado en set agrupado: smote


[I 2026-03-09 05:32:21,699] Trial 0 finished with value: 0.9519828457457737 and parameters: {'weight_func': 'cost_sensitive', 'web_boost': 2.0810221907934716, 'bot_boost': 2.3059655861159816, 'fs_method': 'none', 'n_estimators': 280, 'max_depth': 38, 'min_samples_split': 8, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.9519828457457737.
[I 2026-03-09 05:33:11,399] Trial 1 finished with value: 0.9484151835375338 and parameters: {'weight_func': 'balanced', 'web_boost': 1.516657634659124, 'bot_boost': 1.6039184487965457, 'fs_method': 'none', 'n_estimators': 336, 'max_depth': 49, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.9519828457457737.
[I 2026-03-09 05:33:43,609] Trial 2 finished with value: 0.9113708177226112 and parameters: {'weight_func': 'none', 'fs_method': 'tree', 'k_features': 22, 'n_estimators': 236, 'max_depth': 47, 'min_samples_split': 9, 'min_samples_leaf': 2, 'max_features': 'log2'}.


Resultados finales para smote
Mejor trial: 15
Mejor F1 macro: 0.9611
Mejores params: {'weight_func': 'none', 'fs_method': 'tree', 'k_features': 55, 'n_estimators': 254, 'max_depth': 40, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': 'log2'}


Iniciando estudio enfocado en set agrupado: smote_enn


[I 2026-03-09 06:21:12,011] Trial 0 finished with value: 0.951820956357929 and parameters: {'weight_func': 'none', 'fs_method': 'tree', 'k_features': 58, 'n_estimators': 317, 'max_depth': 47, 'min_samples_split': 9, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 0 with value: 0.951820956357929.
[I 2026-03-09 06:21:48,044] Trial 1 finished with value: 0.9468451303120575 and parameters: {'weight_func': 'cost_sensitive', 'web_boost': 2.9672980281377113, 'bot_boost': 2.1915951985988515, 'fs_method': 'none', 'n_estimators': 257, 'max_depth': 43, 'min_samples_split': 4, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.951820956357929.
[I 2026-03-09 06:22:32,249] Trial 2 finished with value: 0.9500684522801965 and parameters: {'weight_func': 'none', 'fs_method': 'none', 'n_estimators': 314, 'max_depth': 42, 'min_samples_split': 8, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.951820956357929.
[I 2026-03-09 06:23:06,37

In [14]:
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report

print("Prueba final en test set con el mejor modelo")

best_dataset = 'none'
best_trial = 25
best_fs = 'tree'
best_k = 57

x_train_best, y_train_best = train_datasets_grouped[best_dataset]

print(f"Aplicando Feature Selection: {best_fs} con k={best_k}...")
final_selector = FeatureSelector(method=best_fs, k=best_k, class_weight=None)
x_train_final = final_selector.fit_transform(x_train_best, y_train_best)

x_test_final = final_selector.transform(X_test_imp_grouped) 

best_rf_params = {
    'n_estimators': 289,
    'max_depth': 37,
    'min_samples_split': 4,
    'min_samples_leaf': 2,
    'max_features': 'sqrt',
    'class_weight': None,
    'n_jobs': -1,
    'random_state': 42
}

print("Entrenando random forest con los mejores hiperparametros...")
final_rf = RandomForestClassifier(**best_rf_params)
final_rf.fit(x_train_final, y_train_best)

print("Evaluando en el set de test...")
y_pred_test = final_rf.predict(x_test_final)

test_f1_mac = f1_score(y_test_grouped, y_pred_test, average='macro')
test_report = classification_report(y_test_grouped, y_pred_test, zero_division=0)

print("Resultados finales en test set")
print(f"f1 macro: {test_f1_mac:.4f}")
print("\nreporte de clasificacion (test):")
print(test_report)

save_confusion_matrix_new(
    testset="7", 
    y_true=y_test_grouped, 
    y_pred=y_pred_test, 
    trial_number=best_trial, 
    dataset_name=best_dataset, 
    phase="test_final"
)

log_msg = f"""Evaluacion final con test
modelo base: random forest
dataset: {best_dataset} | feature selection: {best_fs} (k={best_k})
params RF: {best_rf_params}

F1 macro (test): {test_f1_mac:.4f}

metricas por clase (test):
{test_report}
"""

log_filename = "Logs_RF_Baseline_7/rf_final_test_metrics.log"
with open(log_filename, 'w', encoding='utf-8') as f:
    f.write(log_msg)

print("\nPipeline de random forest completado con exito.")
print(f"Resultados guardados en: {log_filename}")

Prueba final en test set con el mejor modelo
Aplicando Feature Selection: tree con k=57...


Entrenando random forest con los mejores hiperparametros...
Evaluando en el set de test...
Resultados finales en test set
f1 macro: 0.9014

reporte de clasificacion (test):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    307072
           1       0.81      0.50      0.62       295
           2       1.00      1.00      1.00     19204
           3       1.00      0.99      1.00      1544
           4       1.00      1.00      1.00     34661
           5       0.99      0.99      0.99       825
           6       1.00      0.99      0.99       870
           7       1.00      1.00      1.00      1190
           8       1.00      0.50      0.67         2
           9       0.67      0.40      0.50         5
          10       0.99      1.00      1.00     23840
          11       0.99      1.00      0.99       884
          12       0.98      0.96      0.97       327

    accuracy                           1.00    390719
   macro avg   

In [15]:
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report

best_models_config = {
    'smote': {
        'trial': 15, 
        'fs_method': 'tree', 
        'k_features': 55, 
        'rf_params': {
            'n_estimators': 254, 'max_depth': 40, 'min_samples_split': 5, 
            'min_samples_leaf': 3, 'max_features': 'log2', 'class_weight': None, 
            'n_jobs': -1, 'random_state': 42
        }
    },
    'smote_enn': {
        'trial': 61, 
        'fs_method': 'tree', 
        'k_features': 39, 
        'rf_params': {
            'n_estimators': 290, 'max_depth': 48, 'min_samples_split': 5, 
            'min_samples_leaf': 3, 'max_features': 'log2', 'class_weight': None, 
            'n_jobs': -1, 'random_state': 42
        }
    },
    'smote_tomek': {
        'trial': 52, 
        'fs_method': 'none', 
        'k_features': 80,
        'rf_params': {
            'n_estimators': 225, 'max_depth': 49, 'min_samples_split': 10, 
            'min_samples_leaf': 3, 'max_features': 'log2', 'class_weight': None, 
            'n_jobs': -1, 'random_state': 42
        }
    }
}

print("Iniciando pruebas finales en test set para smote, smote_enn y smote_tomek...\n")

for current_dataset, config in best_models_config.items():
    print(f"Evaluando modelo: {current_dataset.upper()}")
    
    x_train_best, y_train_best = train_datasets_grouped[current_dataset]
    
    print(f"aplicando feature selection: {config['fs_method']} (k={config['k_features']})...")
    final_selector = FeatureSelector(method=config['fs_method'], k=config['k_features'], class_weight=None)
    x_train_final = final_selector.fit_transform(x_train_best, y_train_best)
    x_test_final = final_selector.transform(X_test_imp_grouped) 
    
    print("Entrenando random forest...")
    final_rf = RandomForestClassifier(**config['rf_params'])
    final_rf.fit(x_train_final, y_train_best)
    
    print("Evaluando en el set de test...")
    y_pred_test = final_rf.predict(x_test_final)
    
    test_f1_mac = f1_score(y_test_grouped, y_pred_test, average='macro')
    test_report = classification_report(y_test_grouped, y_pred_test, zero_division=0)
    
    print(f"Resultados finales - F1 macro: {test_f1_mac:.4f}\n")
    
    save_confusion_matrix_new(
        testset="7", 
        y_true=y_test_grouped, 
        y_pred=y_pred_test, 
        trial_number=config['trial'], 
        dataset_name=current_dataset, 
        phase=f"test_final_{current_dataset}"
    )
    
    log_msg = f"""Evaluacion final con test
modelo base: random forest
dataset: {current_dataset} | feature selection: {config['fs_method']} (k={config['k_features']})
params RF: {config['rf_params']}

F1 macro (test): {test_f1_mac:.4f}

Metricas por clase (test):
{test_report}
"""
    log_filename = f"Logs_RF_Baseline_7/rf_final_test_metrics_{current_dataset}.log"
    with open(log_filename, 'w', encoding='utf-8') as f:
        f.write(log_msg)

print("Pipeline de pruebas completado con exito para todos los modelos.")

Iniciando pruebas finales en test set para smote, smote_enn y smote_tomek...

Evaluando modelo: SMOTE
aplicando feature selection: tree (k=55)...
Entrenando random forest...
Evaluando en el set de test...
Resultados finales - F1 macro: 0.9036

Evaluando modelo: SMOTE_ENN
aplicando feature selection: tree (k=39)...
Entrenando random forest...
Evaluando en el set de test...
Resultados finales - F1 macro: 0.8991

Evaluando modelo: SMOTE_TOMEK
aplicando feature selection: none (k=80)...
Entrenando random forest...
Evaluando en el set de test...
Resultados finales - F1 macro: 0.9029

Pipeline de pruebas completado con exito para todos los modelos.
